<a href="https://colab.research.google.com/github/AyushTayal777/Langchain/blob/main/currency_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
from google.colab import userdata

import os
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [25]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental


In [26]:
!pip install langchain_google_genai


In [20]:
import requests
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import InjectedToolArg
from langchain_core.messages import HumanMessage
from typing import Annotated

In [18]:
@tool
def get_conversion_rate(base_currency: str, target_currency: str) -> float:
  """
  this function fetches the currency conversion factor between a base currency and a target curremcy
  """
  url = f'https://v6.exchangerate-api.com/v6/6d7f87ca6002fc58f7f753c9/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
  """
  given a currency rate this function calculates the traget value from a given base currency value
  """

  return base_currency_value*conversion_rate

In [9]:
get_conversion_rate.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1787356801,
 'time_last_update_utc': 'Sat, 22 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1787443201,
 'time_next_update_utc': 'Sun, 23 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.7547}

In [15]:
convert.invoke({'base_currency_value':25,'conversion_rate':95.75})

2393.75

In [27]:
llm=ChatGoogleGenerativeAI(model='gemini-3.6-flash')

In [28]:
llm_with_tools=llm.bind_tools([get_conversion_rate,convert])

In [33]:
messages = [HumanMessage('What is the conversion RATE between INR and USD, and based on that can you convert 10 inr to usd')]

In [34]:
ai_message=llm_with_tools.invoke(messages)

In [36]:
messages.append(ai_message)

In [37]:
ai_message.tool_calls

[{'name': 'get_conversion_rate',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': 'call_1982802',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_1982803',
  'type': 'tool_call'}]

In [42]:
import json

for tool_call in ai_message.tool_calls:
  if tool_call['name']=='get_conversion_rate':
    tool_message1=get_conversion_rate.invoke(tool_call)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    messages.append(tool_message1)

  if tool_call['name']=='convert':
     tool_call['args']['conversion_rate'] = conversion_rate
     tool_message2 = convert.invoke(tool_call)
     messages.append(tool_message2)

In [43]:
messages

[HumanMessage(content='What is the conversion RATE between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'convert', 'arguments': '{"base_currency_value": 10}'}, '__gemini_function_call_thought_signatures__': {'call_1982802': 'EpsTCpgTARFNMg/Idh9y+2x3Gt1/EGHZJ226QgqypU8o137IXSrQ+UqomSHBsEokiHIiWGGc9rcyl/ynI1xq+Y3d6w7mo0p2pOPS+X6Sls5S+qKvgw/h0/bmRrzpCymDleM/NjLdavbveY7HNNKqeNiwFiMsYmKinVdzQQ4/1Fp42MNzAj7O+55Xs6Cuovy3iy8ft/N6MoKvkayxNkW5vIhhbDV4eKDhJhx94gOIbVfn5A02aokPZ0s5kqka8mwIGvplH17G8/dUDnK6q2GOliSyFhNNs+olEZyEfIlloA/jCXLBmsBReiJoLY40zcRnjs6MtILZGqhOwesBEMlJHFibHQvjyO21kGxPDcOxXWsnAa34ZLiJlAZYUkKXwTikgydCg8XKbJP0OQWQH53yRWoFyFgZPKikaxz+JdXfLQfkirMpBhF+usSjebvQIu+5S3zj3py2kK+HzA1v9ryTsvMGeLIuxrwmc49M0NWizQnKIeamUNK1C03AoHKbZ6yhwWeq4W2cwW7Svf547JevijcuhbHmFov5ygDfjj2/plMuM0sQ9p5ZCitxI1XAv/A0jTEJeebUkeNHLPwi6iEVRP8nWvsfyXlkkhYYVi6hENQcEUxgeEEYRMmtk/ffI8fXaJtm

In [44]:
llm_with_tools.invoke(messages).text

'The conversion rate from **INR** to **USD** is **0.01044** (1 INR = 0.01044 USD).\n\nBased on this rate, **10 INR** is equal to **0.1044 USD** (or approximately **$0.10**).'